## Komórka 1: Importy i Globalna Konfiguracja

In [ ]:
from evaluation.post_run_analysis import run_final_evaluation_and_visualization
from model.architecture import GuitarTabCRNN, TabCNN 
from model.utils import load_best_model 
import os 
import json 
import torch 
import training
from IPython.display import display, Markdown 
from torch.utils.data import DataLoader
from IPython.display import clear_output
import config
from training.pipeline import process_single_hyperparameter_run
from data_processing import preparation, batching, dataset
from vizualization import plotting

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_HOME = config.DATA_HOME_DEFAULT
OUTPUT_BASE_DIR = config.OUTPUT_BASE_DIR_DEFAULT
SEARCH_RUN_NAME = "hyperparam_set_v1"
MAIN_SEARCH_ARTIFACTS_DIR = os.path.join(OUTPUT_BASE_DIR, SEARCH_RUN_NAME)

os.makedirs(DATA_HOME, exist_ok=True)
os.makedirs(OUTPUT_BASE_DIR, exist_ok=True)
os.makedirs(MAIN_SEARCH_ARTIFACTS_DIR, exist_ok=True)

print(f"Używany DATA_HOME: {DATA_HOME}")
print(f"Używany OUTPUT_BASE_DIR: {OUTPUT_BASE_DIR}")
print(f"Używany MAIN_SEARCH_ARTIFACTS_DIR: {MAIN_SEARCH_ARTIFACTS_DIR}")

param_grid = []
try:
    with open(config.DEFAULT_HYPERPARAMETER_FILE, 'r', encoding='utf-8') as f:
        param_grid = json.load(f)
    print(f"Wczytano {len(param_grid)} zestawów hiperparametrów z: {config.DEFAULT_HYPERPARAMETER_FILE}")
    if param_grid:
        print(f"Przykładowy pierwszy wczytany zestaw: {param_grid[0]}")
except FileNotFoundError:
    print(f"BŁĄD: Plik z zestawami hiperparametrów '{config.DEFAULT_HYPERPARAMETER_FILE}' nie został znaleziony.")
except json.JSONDecodeError as e_json:
    print(f"BŁĄD: Plik '{config.DEFAULT_HYPERPARAMETER_FILE}' zawiera niepoprawny format JSON. Szczegóły: {e_json}")
except Exception as e:
    print(f"BŁĄD podczas wczytywania zestawów hiperparametrów: {e}")

if not param_grid:
    print("OSTRZEŻENIE: `param_grid` jest pusta. Przeszukiwanie może nie zostać wykonane poprawnie.")

def jupyter_notebook_clear_output(wait=True):
    if config.CLEAR_CONSOLE_EVERY_N_RUNS > 0:
        clear_output(wait=wait)
    else:
        pass


## Komórka 2: Przygotowanie podziału ID utworów i preprocessing danych

In [ ]:
print("--- Etap 1: Przygotowanie i Podział Danych ---")
track_ids_map_split = preparation.prepare_track_splits(
    data_home=DATA_HOME,
    problematic_files_list=config.PROBLEMATIC_FILES,
    test_split_fraction=config.TEST_SPLIT_SIZE,
    validation_split_fraction=config.VALIDATION_SPLIT_SIZE,
    seed=config.RANDOM_SEED,
    output_dir_for_ids=OUTPUT_BASE_DIR 
)

if any(len(ids) > 0 for ids in track_ids_map_split.values()):
    print("\nUruchamianie preprocessingu GuitarSet (pominie istniejące pliki)...")
    preparation.preprocess_guitarset_data(
        guitarset_data_home=DATA_HOME,
        processed_output_base_dir=OUTPUT_BASE_DIR,
        track_ids_map=track_ids_map_split,
        audio_sample_rate=config.SAMPLE_RATE,
        audio_hop_length=config.HOP_LENGTH,
        audio_n_cqt_bins=config.N_BINS_CQT,
        audio_cqt_bins_per_octave=config.BINS_PER_OCTAVE_CQT,
        audio_cqt_fmin=config.FMIN_CQT
    )
    print("Preprocessing zakończony lub pliki już istniały.")
else:
    print("Brak utworów do przetworzenia. Preprocessing nie został uruchomiony.")

## Komórka 3: Tworzenie obiektów Dataset i DataLoader

In [ ]:
print("\n--- Etap 2: Tworzenie Datasetów i DataLoaderów ---")

train_dataset, validation_dataset, test_dataset = None, None, None
train_dataloader, validation_dataloader, test_dataloader = None, None, None

train_dataset_constructor_params = {
    **config.DATASET_COMMON_PARAMS,
    **config.DATASET_TRAIN_AUGMENTATION_PARAMS,
    "label_transform_function": dataset.create_frame_level_labels,
    "guitarset_data_home": DATA_HOME
}

eval_dataset_constructor_params = {
    **config.DATASET_COMMON_PARAMS,
    **config.DATASET_EVAL_AUGMENTATION_PARAMS,
    "label_transform_function": dataset.create_frame_level_labels,
    "guitarset_data_home": DATA_HOME
}

if track_ids_map_split.get('train') and len(track_ids_map_split['train']) > 0:
    train_dataset = dataset.GuitarSetTabDataset(
        processed_data_base_dir=OUTPUT_BASE_DIR,
        data_split_name='train',
        **train_dataset_constructor_params
    )
    if len(train_dataset) > 0:
        train_dataloader = DataLoader(
            train_dataset,
            batch_size=config.BATCH_SIZE_DEFAULT,
            shuffle=True,
            num_workers=0,
            pin_memory=torch.cuda.is_available(),
            collate_fn=batching.collate_fn_pad
        )
        print(f"Utworzono zbiór treningowy: {len(train_dataset)} próbek, DataLoader gotowy.")
    else:
        print("Zbiór treningowy jest pusty.")

if track_ids_map_split.get('validation') and len(track_ids_map_split['validation']) > 0:
    validation_dataset = dataset.GuitarSetTabDataset(
        processed_data_base_dir=OUTPUT_BASE_DIR,
        data_split_name='validation',
        **eval_dataset_constructor_params
    )
    if len(validation_dataset) > 0:
        validation_dataloader = DataLoader(
            validation_dataset,
            batch_size=config.BATCH_SIZE_DEFAULT,
            shuffle=False,
            num_workers=0,
            pin_memory=torch.cuda.is_available(),
            collate_fn=batching.collate_fn_pad
        )
        print(f"Utworzono zbiór walidacyjny: {len(validation_dataset)} próbek, DataLoader gotowy.")
    else:
        print("Zbiór walidacyjny jest pusty.")

if track_ids_map_split.get('test') and len(track_ids_map_split['test']) > 0:
    test_dataset = dataset.GuitarSetTabDataset(
        processed_data_base_dir=OUTPUT_BASE_DIR,
        data_split_name='test',
        **eval_dataset_constructor_params
    )
    if len(test_dataset) > 0:
        test_dataloader = DataLoader(
            test_dataset,
            batch_size=config.BATCH_SIZE_DEFAULT,
            shuffle=False,
            num_workers=0,
            pin_memory=torch.cuda.is_available(),
            collate_fn=batching.collate_fn_pad
        )
        print(f"Utworzono zbiór testowy: {len(test_dataset)} próbek, DataLoader gotowy.")
    else:
        print("Zbiór testowy jest pusty.")

## Komórka 4: Wizualizacja Danych

In [ ]:

print("--- Etap 3: Wizualizacja Próbki Danych (bez augmentacji) ---")

dataset_to_visualize = validation_dataset
if dataset_to_visualize and len(dataset_to_visualize) > 0:
    print("Wybrano zbiór walidacyjny do wizualizacji.")
    sample_idx_to_plot = 0

    try:
        features, (onset_targets, fret_targets), raw_labels, track_id = dataset_to_visualize[sample_idx_to_plot]
        print(f"Wizualizacja próbki: {track_id} (indeks: {sample_idx_to_plot})")
        plotting.plot_sample_data_summary(
            features_tensor=features,
            onset_labels_tensor=onset_targets,
            fret_labels_tensor=fret_targets,
            sampling_rate=config.SAMPLE_RATE,
            hop_len=config.HOP_LENGTH,
            track_id_display=track_id
        )
    except Exception as e:
        print(f"Wystąpił błąd podczas wizualizacji danych: {e}")
else:
    print("Zbiór danych do wizualizacji jest pusty lub niedostępny.")

## Komórka 5: Trening i wizualizacja treningu

In [ ]:


def find_next_run_number(artifacts_dir):
    highest_existing_run = 0
    if os.path.exists(artifacts_dir):
        for item in os.listdir(artifacts_dir):
            if os.path.isdir(os.path.join(artifacts_dir, item)) and item.startswith("run_"):
                try:
                    run_num = int(item.split('_')[1])
                    if run_num > highest_existing_run:
                        highest_existing_run = run_num
                except (IndexError, ValueError):
                    continue
    return highest_existing_run + 1

def display_run_summary(summary_file_path, metric_to_sort_by, num_top_to_show=5):
    if not os.path.exists(summary_file_path):
        print(f"Ostrzeżenie: Plik podsumowania '{summary_file_path}' nie istnieje.")
        return

    all_runs = []
    with open(summary_file_path, "r", encoding="utf-8") as f:
        for line in f:
            try:
                all_runs.append(json.loads(line))
            except json.JSONDecodeError:
                continue

    if not all_runs:
        print("Brak zapisanych przebiegów do wyświetlenia.")
        return

    def get_sortable_value(result_dict):
        primary_metric_val = result_dict.get(metric_to_sort_by, 0.0)
        test_metrics = result_dict.get("test_metrics_at_0.5", {})
        secondary_metric_val = test_metrics.get("tdr_f1", 0.0) if isinstance(test_metrics, dict) else 0.0
        return (primary_metric_val, secondary_metric_val)

    completed_runs = sorted(
        [run for run in all_runs if run.get("status") == "COMPLETED"],
        key=get_sortable_value,
        reverse=True
    )
    error_runs = [run for run in all_runs if run.get("status") != "COMPLETED"]

    print("\n" + "="*25 + " PODSUMOWANIE PRZEBIEGÓW " + "="*25)
    
    print(f"\n--- Najlepsze Przebiegi (Top {min(num_top_to_show, len(completed_runs))}) ---")
    for i, summary in enumerate(completed_runs[:num_top_to_show]):
        print(f"\nPozycja {i + 1}: Przebieg {summary.get('run_index', 'N/A')} (Folder: {summary.get('run_folder_name', 'N/A')})")
        print(f"  - Najlepszy wynik walidacyjny ({metric_to_sort_by}): {summary.get(metric_to_sort_by, 0.0):.4f}")
        
        test_metrics = summary.get('test_metrics_at_0.5', {})
        if isinstance(test_metrics, dict) and test_metrics:
            tdr_f1 = test_metrics.get('tdr_f1', 0)
            mpe_f1 = test_metrics.get('mpe_f1', 0)
            onset_f1 = test_metrics.get('onset_f1_event', 0)
            print(f"  - Metryki na teście (thr=0.5): TDR F1: {tdr_f1:.4f} | MPE F1: {mpe_f1:.4f} | Onset F1: {onset_f1:.4f}")
        else:
            print("  - Brak metryk testowych.")
        print(f"  - Zakończono na epoce: {summary.get('stopped_epoch', 0)}")
        print(f"  - Parametry: {summary.get('params_combo', {})}")

    if error_runs:
        print("\n--- Przebiegi Zakończone Błędem ---")
        for summary in error_runs:
            print(f"\nPrzebieg {summary.get('run_index', 'N/A')} (Folder: {summary.get('run_folder_name', 'N/A')})")
            print(f"  - Status: {summary.get('status', 'N/A')}")
            print(f"  - Błąd: {summary.get('error', 'Brak szczegółów')}")
            print(f"  - Parametry: {summary.get('params_combo', {})}")

print(f"{'=' * 20} ROZPOCZYNANIE PRZESZUKIWANIA HIPERPARAMETRÓW {'=' * 20}")

MAIN_SEARCH_ARTIFACTS_DIR = r"C:\Users\lukig\Documents\Programming\music-transcription\python\results\hyperparam_search"
SUMMARY_FILE_PATH = os.path.join(MAIN_SEARCH_ARTIFACTS_DIR, "hyperparameter_search_summary.jsonl")
current_active_augmentation_params = config.DATASET_TRAIN_AUGMENTATION_PARAMS

os.makedirs(MAIN_SEARCH_ARTIFACTS_DIR, exist_ok=True)
print(f"Główny katalog artefaktów przeszukiwania: {MAIN_SEARCH_ARTIFACTS_DIR}")

if not param_grid:
    print("BŁĄD KRYTYCZNY: `param_grid` jest pusta.")
else:
    next_run_start_number = find_next_run_number(MAIN_SEARCH_ARTIFACTS_DIR)
    print(f"Nowe przebiegi rozpoczną się od numeru (ID): {next_run_start_number}")

    for run_idx, hyperparams in enumerate(param_grid):
        if run_idx > 0 and config.CLEAR_CONSOLE_EVERY_N_RUNS > 0 and run_idx % config.CLEAR_CONSOLE_EVERY_N_RUNS == 0:
            clear_output(wait=True)
            print(f"Output konsoli wyczyszczony. Kontynuacja od przebiegu {run_idx + 1}/{len(param_grid)}...")

        run_global_id = next_run_start_number + run_idx
        print(f"\n\n{'=' * 30} ROZPOCZYNANIE PRZEBIEGU {run_idx + 1}/{len(param_grid)} (Globalne ID: {run_global_id}) {'=' * 30}")
        print(f"Używane hiperparametry: {hyperparams}")

        summary = training.pipeline.process_single_hyperparameter_run(
            run_id=run_global_id,
            hyperparams_combo=hyperparams,
            current_augmentation_params=current_active_augmentation_params,
            config_obj=config,
            main_artifacts_dir=MAIN_SEARCH_ARTIFACTS_DIR,
            train_loader=train_dataloader,
            validation_loader=validation_dataloader,
            test_loader=test_dataloader,
            jupyter_notebook_clear_output_func=clear_output
        )

        with open(SUMMARY_FILE_PATH, "a", encoding="utf-8") as f:
            f.write(json.dumps(summary) + "\n")

        print(f"\n{'=' * 30} ZAKOŃCZONO PRZEBIEG {run_idx + 1}/{len(param_grid)} (ID: {run_global_id}) - Status: {summary.get('status', 'UNKNOWN')} {'=' * 30}\n")

    display_run_summary(SUMMARY_FILE_PATH, f"best_{config.CHECKPOINT_METRIC_DEFAULT}")

## Komórka 6: Generowanie wyników MIDI oraz Tabulatury

In [ ]:
MAIN_SEARCH_ARTIFACTS_DIR = r"C:\Users\lukig\Documents\Programming\music-transcription\python\results\hyperparam_search" 

if 'test_dataset' not in locals() or test_dataset is None:
    print("BŁĄD: Zmienna 'test_dataset' nie jest dostępna.")
else:
    available_runs = [d for d in os.listdir(MAIN_SEARCH_ARTIFACTS_DIR) if d.startswith("run_") and os.path.exists(os.path.join(MAIN_SEARCH_ARTIFACTS_DIR, d, "best_model.pth"))]
    
    if not available_runs:
        print(f"Brak folderów z zapisanymi modelami w '{MAIN_SEARCH_ARTIFACTS_DIR}'")
    else:
        for idx, name in enumerate(available_runs): 
            print(f"[{idx + 1}] {name}")
         
        selected_run_folder_name = None
        while True:
            try:
                choice = int(input(f"\nWybierz numer przebiegu (1-{len(available_runs)}) lub '0' aby pominąć: "))
                if 0 <= choice <= len(available_runs):
                    if choice > 0: 
                        selected_run_folder_name = available_runs[choice - 1]
                    break
                else: 
                    print("Nieprawidłowy wybór.")
            except ValueError: 
                print("Nieprawidłowe dane, podaj liczbę.")

    if selected_run_folder_name:
        display(Markdown(f"## Ewaluacja końcowa dla: `{selected_run_folder_name}`"))
        
        run_dir = os.path.join(MAIN_SEARCH_ARTIFACTS_DIR, selected_run_folder_name)
        config_path = os.path.join(run_dir, "run_configuration.json")
        
        if not os.path.exists(config_path):
             raise FileNotFoundError(f"Brak pliku konfiguracyjnego w {run_dir}.")
        
        with open(config_path, 'r') as f:
            run_params_all = json.load(f)

        key_mapping = {
            'RNN_TYPE': 'rnn_type', 'RNN_HIDDEN_SIZE': 'rnn_hidden_size',
            'RNN_LAYERS': 'rnn_layers', 'RNN_DROPOUT': 'rnn_dropout',
            'RNN_BIDIRECTIONAL': 'rnn_bidirectional'
        }
        params_for_model = {key_mapping[k]: v for k, v in run_params_all.items() if k in key_mapping}
        
        temp_cnn = TabCNN()
        cnn_out_dim = temp_cnn.output_channels * temp_cnn(torch.randn(1, 1, config.N_BINS_CQT, 32)).shape[2]
        model_init_params = {**params_for_model, 'num_frames_rnn_input_dim': cnn_out_dim}
        model_path = os.path.join(run_dir, "best_model.pth")
         
        loaded_model = load_best_model(GuitarTabCRNN, model_init_params, model_path, device)
        if loaded_model is None:
            raise RuntimeError("Nie udało się załadować modelu.")

        output_dir_for_viz = os.path.join(run_dir, "final_test_evaluation_tdr_optimized")
        
        final_metrics, sorted_file_scores = run_final_evaluation_and_visualization(
            model=loaded_model,
            test_dataloader=test_dataloader,
            device=device,
            config_obj=config,
            output_dir=output_dir_for_viz,
            num_top_bottom_files_to_show=3 
        )

        if final_metrics:
            print("\n" + "="*50)
            print("--- PEŁNE PODSUMOWANIE EWALUACJI ---")
            print("="*50)
            display(Markdown(f"**Przebieg:** `{selected_run_folder_name}`"))
            
            print("\n**Uśrednione Metryki na Całym Zbiorze Testowym:**")
            print(f"  TDR F1-Score: {final_metrics.get('tdr_f1', 0):.4f} | TDR Precision: {final_metrics.get('tdr_precision', 0):.4f} | TDR Recall: {final_metrics.get('tdr_recall', 0):.4f}")
            print(f"  MPE F1-Score: {final_metrics.get('mpe_f1', 0):.4f} | MPE Precision: {final_metrics.get('mpe_precision', 0):.4f} | MPE Recall: {final_metrics.get('mpe_recall', 0):.4f}")
            print(f"  Onset Event F1: {final_metrics.get('onset_f1_event', 0):.4f} | Onset Precision: {final_metrics.get('onset_precision_event', 0):.4f} | Onset Recall: {final_metrics.get('onset_recall_event', 0):.4f}")
            
            print("\n**Ranking Plików (pełna lista w plikach .txt w katalogu z artefaktami):**")
        else:
            print("\nEwaluacja nie powiodła się i nie zwróciła metryk. Sprawdź komunikaty o błędach powyżej.")


    else:
        print("\nPominięto ewaluację.")

## Komórka 7: Predykcja na własnych nagraniach

In [ ]:
import inference
import json
from IPython.display import display, Markdown

print("INFO: Ta komórka wykorzystuje bibliotekę 'noisereduce'. Upewnij się, że jest zainstalowana: pip install noisereduce")

MAIN_SEARCH_ARTIFACTS_DIR = r"C:\Users\lukig\Documents\Programming\music-transcription\python\results\hyperparam_search"
selected_run_folder_name_custom = None

available_runs_custom = [d for d in os.listdir(MAIN_SEARCH_ARTIFACTS_DIR) if d.startswith("run_") and os.path.exists(os.path.join(MAIN_SEARCH_ARTIFACTS_DIR, d, "best_model.pth"))]

if not available_runs_custom:
    print(f"BŁĄD: Brak folderów z zapisanymi modelami w '{MAIN_SEARCH_ARTIFACTS_DIR}'")
else:
    for idx, name in enumerate(available_runs_custom):
        print(f"[{idx + 1}] {name}")

    while True:
        try:
            choice = int(input(f"\nWybierz numer przebiegu (1-{len(available_runs_custom)}) do testów na własnych plikach: "))
            if 1 <= choice <= len(available_runs_custom):
                selected_run_folder_name_custom = available_runs_custom[choice - 1]
                break
            else:
                print("Nieprawidłowy wybór.")
        except ValueError:
            print("Nieprawidłowe dane, podaj liczbę.")

if selected_run_folder_name_custom:
    display(Markdown(f"## Testowanie na własnych plikach dla modelu: `{selected_run_folder_name_custom}`"))

    run_dir_custom = os.path.join(MAIN_SEARCH_ARTIFACTS_DIR, selected_run_folder_name_custom)
    config_path_custom = os.path.join(run_dir_custom, "run_configuration.json")
    model_path_custom = os.path.join(run_dir_custom, "best_model.pth")

    if not os.path.exists(config_path_custom):
         raise FileNotFoundError(f"Brak pliku konfiguracyjnego w {run_dir_custom}.")

    with open(config_path_custom, 'r') as f:
        run_config_data = json.load(f)

    hyperparams_source = run_config_data.get('hyperparameters_tuned', run_config_data)

    key_mapping = {
        'RNN_TYPE': 'rnn_type',
        'RNN_HIDDEN_SIZE': 'rnn_hidden_size',
        'RNN_LAYERS': 'rnn_layers',
        'RNN_DROPOUT': 'rnn_dropout',
        'RNN_BIDIRECTIONAL': 'rnn_bidirectional'
    }

    params_for_model_custom = {}
    for json_key, model_key in key_mapping.items():
        if json_key in hyperparams_source:
            params_for_model_custom[model_key] = hyperparams_source[json_key]
    
    temp_cnn = TabCNN()
    cnn_out_dim_custom = temp_cnn.output_channels * temp_cnn(torch.randn(1, 1, config.N_BINS_CQT, 32)).shape[2]
    model_init_params_custom = {**params_for_model_custom, 'num_frames_rnn_input_dim': cnn_out_dim_custom}

    audio_folder_path = r"C:\Users\lukig\Documents\Programming\music-transcription\python\test"
    output_dir_for_custom_preds = r"C:\Users\lukig\Documents\Programming\music-transcription\python\test"
    
    print(f"\nFolder z plikami audio: {audio_folder_path}")
    print(f"Folder zapisu tabulatur: {output_dir_for_custom_preds}")

    if not os.path.isdir(audio_folder_path):
        print("UWAGA: Podana ścieżka do plików audio nie istnieje lub nie jest folderem.")

    inference.run_inference_on_directory(
        model_class=GuitarTabCRNN,
        model_init_params=model_init_params_custom,
        model_path=model_path_custom,
        audio_dir=audio_folder_path,
        output_dir=output_dir_for_custom_preds,
        device=device,
        config_obj=config
    )
else:
    print("\nNie wybrano modelu, pominięto testowanie na własnych plikach.")